# 03 · HMM / Regime Switching — Finance Concept

**Contexto:** Los Quant Macro (Renaissance Technologies, Bridgewater) usan HMMs para detectar automáticamente regímenes de mercado — bull, bear, alta volatilidad — sin definir las fechas manualmente. Los parámetros del modelo de trading cambian según el régimen activo.

**Campo de origen:** Quant Macro · Regime Switching · Hamilton (1989)  
**Dataset:** S&P 500 retornos diarios — simulado con estadísticos reales de los regímenes históricos

> Para datos reales: `yfinance.download('^GSPC', start='2000-01-01')`

---

## Marco teórico

### El modelo HMM-Gaussiano

El mercado alterna entre $K$ estados ocultos $S_t \in \{1, \ldots, K\}$ con dinámica de Markov:

$$P(S_t = j \mid S_{t-1} = i) = a_{ij} \qquad \sum_j a_{ij} = 1$$

Cada estado genera retornos con su propia distribución:

$$r_t \mid S_t = k \;\sim\; \mathcal{N}(\mu_k,\, \sigma_k^2)$$

### Los tres algoritmos

| Algoritmo | Pregunta | Uso |
|-----------|----------|-----|
| **Forward (filtrado)** | $P(S_t=k \mid r_1,\ldots,r_t)$ | Régimen actual en tiempo real |
| **Viterbi (decodificación)** | $\arg\max P(S_1^*,\ldots,S_T^* \mid r)$ | Etiqueta histórica de cada día |
| **Baum-Welch (EM)** | $\hat{\lambda} = \arg\max P(r \mid \lambda)$ | Aprender parámetros de los datos |

### Duración esperada de cada régimen

$$\mathbb{E}[\text{duración estado } k] = \frac{1}{1 - a_{kk}}$$

**Referencias:** Hamilton, J.D. (1989). A new approach to the economic analysis of nonstationary time series. *Econometrica* 57(2). Baum et al. (1970). *Ann. Math. Stat.* 41(1).

In [ ]:
# ── IMPORTS ───────────────────────────────────────────────────────────────────
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import norm
warnings.filterwarnings('ignore')
os.makedirs('data', exist_ok=True)

C = dict(
    price='#1E293B',  r1='#15803D',   r2='#B91C1C',   r3='#F59E0B',
    prob1='#2563EB',  prob2='#DC2626', prob3='#F59E0B',
    fill1='#DCFCE7',  fill2='#FEE2E2', fill3='#FEF9C3',
    neutral='#64748B'
)
np.random.seed(42)
print('✓ OK')

In [ ]:
# ── DATOS — S&P 500 retornos diarios ─────────────────────────────────────────
# Fuente real: yf.download('^GSPC', start='2000-01-01')['Close'].pct_change()
#
# Simulación con 3 regímenes históricos documentados del S&P 500:
#   Régimen 1 (Bull):       μ=+0.08%/día  σ=0.60%  → Sharpe ≈ 2.1 anual
#   Régimen 2 (Bear):       μ=-0.15%/día  σ=1.80%  → caídas sostenidas
#   Régimen 3 (Alta Vol):   μ=+0.02%/día  σ=2.50%  → crisis/recuperación
#
# Matriz de transición aproximada (Hamilton 1989 + estimaciones empíricas):
#   A = [[0.97, 0.02, 0.01],
#        [0.03, 0.94, 0.03],
#        [0.05, 0.05, 0.90]]

n = 1500  # ~6 años de días hábiles
dates = pd.bdate_range('2018-01-02', periods=n)

# Parámetros por régimen
params = {
    1: {'mu': 0.0008,  'sigma': 0.006},   # Bull
    2: {'mu': -0.0015, 'sigma': 0.018},   # Bear
    3: {'mu': 0.0002,  'sigma': 0.025},   # Alta volatilidad
}
A = np.array([[0.97, 0.02, 0.01],
              [0.03, 0.94, 0.03],
              [0.05, 0.05, 0.90]])

# Simular secuencia de estados y retornos
true_states, returns = [], []
state = 1
for t in range(n):
    true_states.append(state)
    r = np.random.normal(params[state]['mu'], params[state]['sigma'])
    returns.append(r)
    state = np.random.choice([1, 2, 3], p=A[state - 1])

df = pd.DataFrame({
    'return'     : returns,
    'true_state' : true_states,
    'price'      : 2700 * np.cumprod(1 + np.array(returns))
}, index=dates)

state_counts = pd.Series(true_states).value_counts().sort_index()
print('Distribución de regímenes simulados:')
labels = {1:'Bull', 2:'Bear', 3:'Alta Vol'}
for s, cnt in state_counts.items():
    dur = 1 / (1 - A[s-1, s-1])
    print(f'  Estado {s} ({labels[s]}): {cnt} días ({cnt/n:.1%}) · duración esperada {dur:.0f} días')

## Mini-EDA

In [ ]:
# ── EDA 1/2 — Estadísticos clave ─────────────────────────────────────────────
r = df['return']
print(f'{"Métrica":<25} {"Valor":<15} Nota')
print('─' * 65)
rows = [
    ('n días',            len(df),                    '~6 años'),
    ('Retorno μ diario',  f'{r.mean():.4%}',           f'anual ≈ {r.mean()*252:.1%}'),
    ('Vol diaria',        f'{r.std():.4%}',            f'anual ≈ {r.std()*np.sqrt(252):.1%}'),
    ('Skewness',          f'{r.skew():.3f}',           'negativo → cola izquierda (crashes)'),
    ('Kurtosis',          f'{r.kurt():.3f}',           '> 0 → fat tails'),
    ('% días positivos',  f'{(r>0).mean():.1%}',       ''),
    ('VaR 1% diario',     f'{r.quantile(0.01):.3%}',   'pérdida máxima al 99% confianza'),
    ('Max retorno',       f'{r.max():.3%}',            ''),
    ('Min retorno',       f'{r.min():.3%}',            '(crash)'),
]
for label, val, note in rows:
    print(f'{label:<25} {str(val):<15} {note}')

In [ ]:
# ── EDA 2/2 — Precio + distribución de retornos ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4), gridspec_kw={'width_ratios': [3, 1]})

ax = axes[0]
ax.plot(df.index, df.price, color=C['price'], lw=0.9)
# Colorear fondo por régimen verdadero
state_colors = {1: C['fill1'], 2: C['fill2'], 3: C['fill3']}
prev_state, prev_date = df.true_state.iloc[0], df.index[0]
for i in range(1, len(df)):
    if df.true_state.iloc[i] != prev_state or i == len(df)-1:
        ax.axvspan(prev_date, df.index[i],
                   alpha=0.3, color=state_colors[prev_state], lw=0)
        prev_state, prev_date = df.true_state.iloc[i], df.index[i]
ax.set_title('S&P 500 — precio con regímenes verdaderos (referencia)', fontsize=10)
ax.set_ylabel('Precio USD')
ax.grid(axis='y', alpha=0.3)
# Leyenda
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color=C['fill1'], alpha=0.5, label='Bull'),
    Patch(color=C['fill2'], alpha=0.5, label='Bear'),
    Patch(color=C['fill3'], alpha=0.5, label='Alta Vol'),
], fontsize=8, loc='upper left')

ax2 = axes[1]
ax2.hist(df['return']*100, bins=80, color=C['fill1'],
         edgecolor=C['price'], lw=0.3, density=True, orientation='horizontal')
x = np.linspace(df['return'].min()*100, df['return'].max()*100, 200)
ax2.plot(norm.pdf(x, df['return'].mean()*100, df['return'].std()*100),
         x, color=C['price'], lw=1.0, ls='--', label='Normal')
ax2.set_title('Dist. retornos\n(fat tails visibles)', fontsize=10)
ax2.set_xlabel('Densidad')
ax2.legend(fontsize=8)
ax2.grid(axis='x', alpha=0.3)

plt.suptitle('Mini-EDA — S&P 500 simulado 3 regímenes', fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig('data/finance_eda.png', dpi=130, bbox_inches='tight')
plt.show()
print('✓ data/finance_eda.png')

In [ ]:
# ── HMM — IMPLEMENTACIÓN MANUAL ──────────────────────────────────────────────
# Implementamos los 3 algoritmos desde cero para transparencia total.
# En producción usar: hmmlearn (pip install hmmlearn)

class GaussianHMM:
    """
    HMM con emisiones Gaussianas.
    Implementa Forward, Viterbi y Baum-Welch (EM).
    """

    def __init__(self, K=2, n_iter=100, tol=1e-4, random_state=42):
        self.K = K
        self.n_iter = n_iter
        self.tol = tol
        self.rs = random_state

    def _emission(self, x, k):
        """Densidad Gaussiana del estado k en el punto x."""
        return norm.pdf(x, self.mu[k], self.sigma[k])

    def _forward(self, obs):
        """Algoritmo Forward — retorna alpha y log-verosimilitud."""
        T, K = len(obs), self.K
        alpha = np.zeros((T, K))
        scale = np.zeros(T)  # factores de escala para estabilidad numérica

        # Inicialización
        alpha[0] = self.pi * np.array([self._emission(obs[0], k) for k in range(K)])
        scale[0] = alpha[0].sum()
        alpha[0] /= scale[0] + 1e-300

        # Recursión
        for t in range(1, T):
            emit = np.array([self._emission(obs[t], k) for k in range(K)])
            alpha[t] = (alpha[t-1] @ self.A) * emit
            scale[t] = alpha[t].sum()
            alpha[t] /= scale[t] + 1e-300

        log_lik = np.sum(np.log(scale + 1e-300))
        return alpha, scale, log_lik

    def _backward(self, obs, scale):
        """Algoritmo Backward."""
        T, K = len(obs), self.K
        beta = np.zeros((T, K))
        beta[-1] = 1.0

        for t in range(T-2, -1, -1):
            emit = np.array([self._emission(obs[t+1], k) for k in range(K)])
            beta[t] = (self.A * emit) @ beta[t+1]
            beta[t] /= scale[t+1] + 1e-300

        return beta

    def fit(self, obs):
        """Baum-Welch (EM) para estimar A, mu, sigma, pi."""
        np.random.seed(self.rs)
        K, T = self.K, len(obs)

        # Inicialización con k-means simple
        indices = np.argsort(obs)
        chunk = T // K
        self.mu    = np.array([obs[indices[k*chunk:(k+1)*chunk]].mean() for k in range(K)])
        self.sigma = np.array([obs[indices[k*chunk:(k+1)*chunk]].std() + 1e-6 for k in range(K)])
        self.pi    = np.ones(K) / K
        self.A     = np.full((K, K), 1/K)
        np.fill_diagonal(self.A, 0.8)
        self.A /= self.A.sum(axis=1, keepdims=True)

        prev_log_lik = -np.inf
        self.log_liks = []

        for iteration in range(self.n_iter):
            # ── Paso E ──
            alpha, scale, log_lik = self._forward(obs)
            beta  = self._backward(obs, scale)

            # gamma: P(S_t=k | obs)
            gamma = alpha * beta
            gamma /= gamma.sum(axis=1, keepdims=True) + 1e-300

            # xi: P(S_t=i, S_{t+1}=j | obs)
            xi = np.zeros((T-1, K, K))
            for t in range(T-1):
                emit_next = np.array([self._emission(obs[t+1], k) for k in range(K)])
                xi[t] = (alpha[t:t+1].T * self.A) * (emit_next * beta[t+1])
                xi[t] /= xi[t].sum() + 1e-300

            # ── Paso M ──
            self.pi    = gamma[0]
            self.A     = xi.sum(axis=0) / (gamma[:-1].sum(axis=0, keepdims=True).T + 1e-300)
            self.A    /= self.A.sum(axis=1, keepdims=True) + 1e-300
            self.mu    = (gamma * obs[:, None]).sum(axis=0) / (gamma.sum(axis=0) + 1e-300)
            self.sigma = np.sqrt(
                (gamma * (obs[:, None] - self.mu)**2).sum(axis=0) /
                (gamma.sum(axis=0) + 1e-300)
            ) + 1e-6

            self.log_liks.append(log_lik)
            if abs(log_lik - prev_log_lik) < self.tol:
                print(f'  Convergencia en iteración {iteration+1}')
                break
            prev_log_lik = log_lik

        self.gamma_ = gamma
        return self

    def predict_proba(self, obs):
        """Retorna P(S_t=k | obs_1..obs_t) — filtrado Forward."""
        alpha, _, _ = self._forward(obs)
        return alpha

    def predict(self, obs):
        """Viterbi — secuencia de estados más probable."""
        T, K = len(obs), self.K
        delta = np.zeros((T, K))
        psi   = np.zeros((T, K), dtype=int)

        delta[0] = np.log(self.pi + 1e-300) + \
                   np.array([np.log(self._emission(obs[0], k) + 1e-300) for k in range(K)])

        for t in range(1, T):
            emit = np.array([np.log(self._emission(obs[t], k) + 1e-300) for k in range(K)])
            for j in range(K):
                trans = delta[t-1] + np.log(self.A[:, j] + 1e-300)
                psi[t, j]   = np.argmax(trans)
                delta[t, j] = trans[psi[t, j]] + emit[j]

        states = np.zeros(T, dtype=int)
        states[-1] = np.argmax(delta[-1])
        for t in range(T-2, -1, -1):
            states[t] = psi[t+1, states[t+1]]

        return states + 1  # estados 1-indexed


print('Clase GaussianHMM definida ✓')

In [ ]:
# ── ENTRENAR HMM — S&P 500 ────────────────────────────────────────────────────
obs = df['return'].values

# Comparar K=2 vs K=3 usando BIC
# BIC = -2·log_lik + n_params·log(T)
# n_params HMM-K: K² (A) + K (mu) + K (sigma) + K (pi) = K²+3K

results = {}
for K in [2, 3]:
    hmm = GaussianHMM(K=K, n_iter=150, tol=1e-5, random_state=42)
    hmm.fit(obs)
    _, _, log_lik = hmm._forward(obs)
    n_params = K**2 + 3*K
    bic = -2 * log_lik + n_params * np.log(len(obs))
    results[K] = {'model': hmm, 'log_lik': log_lik, 'bic': bic}
    print(f'K={K}: log_lik={log_lik:.1f}  BIC={bic:.1f}  n_params={n_params}')

best_K = min(results, key=lambda k: results[k]['bic'])
hmm = results[best_K]['model']
print(f'\nMejor modelo: K={best_K} (BIC más bajo)')

# Ordenar estados por media (estado 1 = más negativo = Bear)
order = np.argsort(hmm.mu)
hmm.mu    = hmm.mu[order]
hmm.sigma = hmm.sigma[order]
hmm.pi    = hmm.pi[order]
hmm.A     = hmm.A[np.ix_(order, order)]
hmm.gamma_ = hmm.gamma_[:, order]

print('\n── Parámetros estimados ──────────────────────────────')
regime_names = ['Bear', 'Bull'] if best_K == 2 else ['Bear', 'Normal', 'Bull']
for k in range(best_K):
    dur = 1 / (1 - hmm.A[k, k])
    print(f'Estado {k+1} ({regime_names[k]}): μ={hmm.mu[k]:.4%}  σ={hmm.sigma[k]:.4%}  dur≈{dur:.0f}d')

print('\nMatriz de transición A:')
print(pd.DataFrame(hmm.A.round(3),
                   index=[f'Desde {n}' for n in regime_names],
                   columns=[f'Hacia {n}' for n in regime_names]).to_string())

In [ ]:
# ── DECODIFICACIÓN Y PROBABILIDADES ─────────────────────────────────────────
viterbi_states = hmm.predict(obs)
filter_probs   = hmm.predict_proba(obs)
smoothed_probs = hmm.gamma_

df['viterbi']   = viterbi_states
for k in range(best_K):
    df[f'prob_{k+1}'] = filter_probs[:, k]
    df[f'smooth_{k+1}'] = smoothed_probs[:, k]

# Precisión vs. estados verdaderos (solo válido porque simulamos los datos)
# Mapear estados estimados a estados verdaderos por mayor coincidencia
from itertools import permutations
if best_K == 2:
    true_bin = np.where(df.true_state == 1, 1, 2)  # Bull=1, resto=2
    acc = (df.viterbi == true_bin).mean()
    print(f'Precisión Viterbi vs. régimen real (K=2, aproximado): {acc:.1%}')
print(f'\nDistribución Viterbi:\n{pd.Series(viterbi_states).value_counts().sort_index().to_string()}')

In [ ]:
# ── DASHBOARD ─────────────────────────────────────────────────────────────────
K = best_K
regime_colors_fill = [C['fill2'], C['fill1']] if K==2 else [C['fill2'], C['fill3'], C['fill1']]
regime_colors_line = [C['r2'],    C['r1']   ] if K==2 else [C['r2'],    C['r3'],    C['r1']   ]

fig = plt.figure(figsize=(15, 12))
fig.suptitle(
    f'HMM Regime Detection — S&P 500 · K={K} estados\n'
    'Quant Macro: detectar bull/bear sin definir fechas manualmente',
    fontsize=13, fontweight='bold', y=0.99
)
gs = gridspec.GridSpec(4, 1, hspace=0.08, height_ratios=[2.5, 1, 1, 1])

# Panel 1 — Precio + regímenes Viterbi
ax1 = fig.add_subplot(gs[0])
ax1.plot(df.index, df.price, color=C['price'], lw=0.9, zorder=3)
prev_s, prev_d = df.viterbi.iloc[0], df.index[0]
for i in range(1, len(df)):
    if df.viterbi.iloc[i] != prev_s or i == len(df)-1:
        ax1.axvspan(prev_d, df.index[i],
                    alpha=0.25, color=regime_colors_fill[prev_s-1], lw=0)
        prev_s, prev_d = df.viterbi.iloc[i], df.index[i]
ax1.set_ylabel('Precio USD')
ax1.set_title(f'Panel 1 — Precio + Regímenes Viterbi (K={K})', loc='left', fontsize=10, pad=5)
from matplotlib.patches import Patch
ax1.legend(handles=[
    Patch(color=regime_colors_fill[k], alpha=0.5, label=regime_names[k])
    for k in range(K)
], fontsize=9, loc='upper left')
ax1.grid(axis='y', alpha=0.3)
ax1.set_xticklabels([])

# Panel 2 — Retornos diarios coloreados por régimen
ax2 = fig.add_subplot(gs[1], sharex=ax1)
for k in range(1, K+1):
    mask = df.viterbi == k
    ax2.bar(df.index[mask], df['return'].values[mask]*100,
            color=regime_colors_line[k-1], alpha=0.7, width=0.8)
ax2.axhline(0, color=C['neutral'], lw=0.6)
ax2.set_ylabel('Retorno (%)')
ax2.set_title('Panel 2 — Retornos diarios por régimen', loc='left', fontsize=10, pad=4)
ax2.grid(axis='y', alpha=0.3)
ax2.set_xticklabels([])

# Panel 3 — Probabilidades filtradas
ax3 = fig.add_subplot(gs[2], sharex=ax1)
prob_colors = [C['prob2'], C['prob1']] if K==2 else [C['prob2'], C['prob3'], C['prob1']]
for k in range(K):
    ax3.plot(df.index, df[f'prob_{k+1}'],
             color=prob_colors[k], lw=0.9, label=regime_names[k])
ax3.axhline(0.7, color=C['neutral'], lw=0.7, ls='--', alpha=0.7, label='Umbral 70%')
ax3.set_ylim(-0.05, 1.05)
ax3.set_ylabel('P(régimen)')
ax3.set_title('Panel 3 — Probabilidades filtradas (Forward)', loc='left', fontsize=10, pad=4)
ax3.legend(fontsize=8, loc='upper right')
ax3.grid(axis='y', alpha=0.3)
ax3.set_xticklabels([])

# Panel 4 — Log-verosimilitud durante entrenamiento
ax4 = fig.add_subplot(gs[3])
ax4.plot(hmm.log_liks, color=C['r1'], lw=1.2)
ax4.set_xlabel('Iteración Baum-Welch')
ax4.set_ylabel('Log-Lik')
ax4.set_title('Panel 4 — Convergencia Baum-Welch (EM)', loc='left', fontsize=10, pad=4)
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('data/finance_dashboard.png', dpi=140, bbox_inches='tight')
plt.show()
print('✓ data/finance_dashboard.png')

In [ ]:
# ── ANÁLISIS — Performance por régimen ──────────────────────────────────────
print('── Estadísticos de retorno por régimen Viterbi ───────────────────')
for k in range(1, K+1):
    r_reg = df.loc[df.viterbi==k, 'return']
    sharpe = r_reg.mean() / r_reg.std() * np.sqrt(252)
    print(f'Estado {k} ({regime_names[k-1]}): n={len(r_reg)}d  '
          f'μ={r_reg.mean():.4%}  σ={r_reg.std():.4%}  '
          f'Sharpe={sharpe:.2f}  VaR1%={r_reg.quantile(0.01):.3%}')

print('\n── Señal de trading basada en régimen ───────────────────────────')
print('Estrategia: long solo en régimen Bull, cash en Bear/Alta Vol')
df['signal']     = (df.viterbi == K).astype(int)  # long en último estado (Bull)
df['ret_strategy'] = df['return'] * df['signal'].shift(1)
df['equity_strat'] = (1 + df['ret_strategy'].fillna(0)).cumprod() * 100
df['equity_bh']    = (1 + df['return']).cumprod() * 100

sr_s = df['ret_strategy'].mean() / df['ret_strategy'].std() * np.sqrt(252)
sr_b = df['return'].mean() / df['return'].std() * np.sqrt(252)
print(f'  Estrategia HMM: Sharpe={sr_s:.2f}  Retorno total={(df.equity_strat.iloc[-1]/100-1):.1%}')
print(f'  Buy & Hold:     Sharpe={sr_b:.2f}  Retorno total={(df.equity_bh.iloc[-1]/100-1):.1%}')

In [ ]:
# ── EXPORTAR ─────────────────────────────────────────────────────────────────
df.to_csv('data/finance_hmm_output.csv')
print('✓ data/finance_hmm_output.csv')
print('✓ data/finance_eda.png')
print('✓ data/finance_dashboard.png')

## Conclusiones — contexto financiero

| Concepto | En mercados | Aplicación Supply Chain |
|----------|------------|------------------------|
| Estado oculto | Régimen bull/bear/alta vol | Régimen baja/campaña/crisis |
| Matriz A | Persistencia de tendencias | Duración esperada de temporada |
| Forward | Régimen actual en tiempo real | ¿Estamos en campaña ahora? |
| Viterbi | Etiqueta histórica de cada día | Etiqueta histórica de cada semana |
| Baum-Welch | Aprende parámetros de retornos | Aprende μ/σ de cada régimen de demanda |
| BIC | Elegir K óptimo | Elegir K óptimo |

**Próximo:** `2_Supply_Adaptation.ipynb` — mismo modelo sobre demanda semanal real del UCI Online Retail dataset.